# FusionCore v0 — Phase 4d: SOTA Training — NHITS

**Notebook:** `04d_SOTA_NHITS.ipynb`  
**Phase:** 4 of 5 (Part D)  
**Objective:** Train NHITS with literature-standard hyperparameters.

**Input:** Phase 4a NN feature matrices (87 features after T2/T3/T5).  
**Output:** NHITS checkpoint, validation predictions.

---

### References

- **NHITS:** Challu, C. et al. (2023). *NHITS: Neural Hierarchical Interpolation for Time Series Forecasting.* AAAI.
- **PatchTST note:** PatchTST (Nie et al. 2023) was originally planned but does not support
  exogenous variables in NeuralForecast (`EXOGENOUS_FUTR = False, EXOGENOUS_HIST = False`).
  NHITS provides the same NeuralForecast integration with full `hist_exog_list` support
  and MLP-based architecture diversity vs RNN (DeepAR) and Transformer (TFT).


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1 — Environment Setup (Run First)
# ══════════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2 — Dependency Installation
# ══════════════════════════════════════════════════════════════════════════════

%%capture
!pip install neuralforecast

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3 — Project Constants, Imports & FusionCore Palette
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import gc

DRIVE_ROOT     = Path('/content/drive/MyDrive/PI')
OUTPUTS_DIR    = DRIVE_ROOT / 'FusionCore' / 'v0' / 'outputs'
CHECKPOINT_DIR = DRIVE_ROOT / 'FusionCore' / 'v0' / 'checkpoints'

RUL_CAP        = 125
RANDOM_STATE   = 42
UNIT_KEY       = ['subset_origin', 'unit_id']

np.random.seed(RANDOM_STATE)

import torch
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.set_float32_matmul_precision('medium')

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from sklearn.metrics import mean_squared_error, mean_absolute_error

FC_DARK_BLUE  = '#0D1B2A'
FC_NAVY       = '#1B3A5C'
FC_ORANGE     = '#D96A1B'
FC_DEEP_RED   = '#9B1B30'
FC_STEEL      = '#4A6274'
FC_CHARCOAL   = '#2D2D2D'
FC_LIGHT_GREY = '#E8E8E8'

plt.rcParams.update({
    'figure.figsize': (14, 5), 'figure.dpi': 150, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'axes.titlesize': 13, 'axes.labelsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
})

def compute_nasa_score(y_true, y_pred):
    d = y_pred - y_true
    return float(np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)))

print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4 — Load Phase 4a Artefacts
# ══════════════════════════════════════════════════════════════════════════════

X_train_nn = pd.read_parquet(OUTPUTS_DIR / 'X_train_nn.parquet')
X_val_nn   = pd.read_parquet(OUTPUTS_DIR / 'X_val_nn.parquet')
y_train    = pd.read_parquet(OUTPUTS_DIR / 'y_train.parquet').squeeze()
y_val      = pd.read_parquet(OUTPUTS_DIR / 'y_val.parquet').squeeze()
meta_train = pd.read_parquet(OUTPUTS_DIR / 'meta_train.parquet')
meta_val   = pd.read_parquet(OUTPUTS_DIR / 'meta_val.parquet')

nn_feature_names = joblib.load(OUTPUTS_DIR / 'nn_feature_names.pkl')

print(f'X_train_nn: {X_train_nn.shape}')
print(f'X_val_nn:   {X_val_nn.shape}')
print(f'NN features: {len(nn_feature_names)}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5 — Build NeuralForecast DataFrames
# ══════════════════════════════════════════════════════════════════════════════

def build_nf_df(X_nn, y, meta):
    """Build NeuralForecast format DataFrame: unique_id, ds, y + exogenous."""
    df = pd.concat([
        meta.reset_index(drop=True),
        X_nn.reset_index(drop=True),
        y.reset_index(drop=True).to_frame('y'),
    ], axis=1)
    df['unique_id'] = df['subset_origin'].astype(str) + '_' + df['unit_id'].astype(str)
    df = df.rename(columns={'cycle': 'ds'})
    df = df.sort_values(['unique_id', 'ds']).reset_index(drop=True)
    # Keep only NeuralForecast columns.
    nf_cols = ['unique_id', 'ds', 'y'] + nn_feature_names
    return df[nf_cols]

train_nf = build_nf_df(X_train_nn, y_train, meta_train)
val_nf   = build_nf_df(X_val_nn, y_val, meta_val)

# ── Truncated evaluation DataFrame ───────────────────────────────────────
# Run-to-failure validation engines always end at RUL=0.  Truncating at
# random cycles produces diverse true RUL values for meaningful evaluation.
np.random.seed(RANDOM_STATE)
eval_rows = []
for uid, grp in val_nf.groupby('unique_id'):
    grp = grp.sort_values('ds')
    n = len(grp)
    lo = min(20, n)
    cutoff = np.random.randint(lo, n + 1) if n > lo else n
    eval_rows.append(grp.iloc[:cutoff])

eval_nf = pd.concat(eval_rows, ignore_index=True)
eval_true_rul = (
    eval_nf.sort_values(['unique_id', 'ds'])
    .groupby('unique_id')['y'].last()
)

print(f'Train NF: {train_nf.shape} ({train_nf["unique_id"].nunique()} engines)')
print(f'Val NF:   {val_nf.shape} ({val_nf["unique_id"].nunique()} engines)')
print(f'Eval NF:  {eval_nf.shape} (truncated trajectories)')
print(f'  True RUL range: [{eval_true_rul.min():.0f}, {eval_true_rul.max():.0f}]')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6 — NHITS Hyperparameters (Literature-Standard, Capacity-Scaled)
# ══════════════════════════════════════════════════════════════════════════════
# Hyperparameters from Challu et al. (2023), capacity-scaled for C-MAPSS.
# MLP widths reduced from 256 to 64 to match the 567-engine training set.
# Three stacks with pooling kernels [4,2,1] and frequency downsampling
# [4,2,1] capture degradation at coarse, medium, and fine temporal scales.

NHITS_PARAMS = {
    'input_size':    50,
    'n_blocks':      [1, 1, 1],
    'mlp_units':     [[64, 64], [64, 64], [64, 64]],
    'n_pool_kernel_size': [4, 2, 1],
    'n_freq_downsample':  [4, 2, 1],
}

print('NHITS hyperparameters (capacity-scaled for C-MAPSS):')
for k, v in NHITS_PARAMS.items():
    print(f'  {k}: {v}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7 — Train NHITS (NeuralForecast)
# ══════════════════════════════════════════════════════════════════════════════
# Key configuration:
#   - exclude_insample_y=True: prevents the model from seeing past RUL
#     values, forcing it to learn exclusively from sensor features.
#   - scaler_type='identity': disables per-engine TemporalNorm so the model
#     sees absolute RUL values.  Phase 2 already provides physics-grounded
#     per-regime normalisation of the sensor inputs.
#   - loss=MSE(): optimises for the conditional mean rather than the median.
#     MSE penalises large errors quadratically, forcing the model to
#     discriminate across the full 0–125 RUL range.
#   - learning_rate=1e-4: reduced from 1e-3 to prevent gradient explosion
#     under MSE loss (targets up to 125 produce large squared-error
#     gradients that overflow at higher learning rates).
#   - dropout_prob_theta=0.3: regularisation applied to the MLP output
#     (theta) coefficients within each NHITS stack.
#   - val_size=20 + early_stop_patience_steps=500: holds out the last 20
#     cycles of each training engine for validation-based early stopping.
#   - Training must complete naturally — manual interruption leaves the
#     model in an unfitted state that blocks .predict() calls.

from neuralforecast.losses.pytorch import MSE

model = NHITS(
    h=1,
    input_size=NHITS_PARAMS['input_size'],
    n_blocks=NHITS_PARAMS['n_blocks'],
    mlp_units=NHITS_PARAMS['mlp_units'],
    n_pool_kernel_size=NHITS_PARAMS['n_pool_kernel_size'],
    n_freq_downsample=NHITS_PARAMS['n_freq_downsample'],
    dropout_prob_theta=0.3,
    loss=MSE(),
    hist_exog_list=nn_feature_names,
    exclude_insample_y=True,
    scaler_type='identity',
    max_steps=5000,
    early_stop_patience_steps=500,
    val_check_steps=50,
    batch_size=64,
    step_size=1,
    learning_rate=1e-4,
    random_seed=RANDOM_STATE,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
)

print('Training NHITS (MSE loss, mlp=64, lr=1e-4)...')
nf_best = NeuralForecast(models=[model], freq=1)
nf_best.fit(df=train_nf, val_size=20)

ckpt_path = str(CHECKPOINT_DIR / 'nhits_best')
nf_best.save(ckpt_path, overwrite=True)
print(f'✔ NHITS checkpoint saved to {ckpt_path}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8 — Validation Predictions (Truncated Trajectory Evaluation)
# ══════════════════════════════════════════════════════════════════════════════

# Predict on truncated trajectories.  NHITS uses hist_exog_list, so
# sensor features are provided as historical context via the df parameter.
preds = nf_best.predict(df=eval_nf).reset_index()
pred_col = [c for c in preds.columns if c not in ['unique_id', 'ds']][0]

last_preds = preds.sort_values(['unique_id', 'ds']).groupby('unique_id').last()
common = last_preds.index.intersection(eval_true_rul.index)

y_pred_nhits = last_preds.loc[common, pred_col].values
y_true = eval_true_rul.loc[common].values

rmse = float(np.sqrt(mean_squared_error(y_true, y_pred_nhits)))
mae  = float(mean_absolute_error(y_true, y_pred_nhits))
nasa = compute_nasa_score(y_true, y_pred_nhits)

print(f'NHITS Validation Results (Truncated Trajectory Evaluation)')
print(f'  RMSE:       {rmse:.4f}')
print(f'  MAE:        {mae:.4f}')
print(f'  NASA Score: {nasa:,.1f}')
print(f'  Predictions: {len(common)} engines')
print(f'  True RUL range: [{y_true.min():.0f}, {y_true.max():.0f}]')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9 — Validation Diagnostics
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(y_true, y_pred_nhits, c=FC_STEEL, s=20, alpha=0.6)
lims = [0, RUL_CAP + 5]
axes[0].plot(lims, lims, '--', color=FC_CHARCOAL, linewidth=1, alpha=0.6)
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel('True RUL (cycles)')
axes[0].set_ylabel('Predicted RUL (cycles)')
axes[0].set_title(f'NHITS \u2014 Predicted vs Actual (RMSE={rmse:.2f})')

residuals = y_pred_nhits - y_true
axes[1].hist(residuals, bins=40, color=FC_STEEL, alpha=0.7, edgecolor='white')
axes[1].axvline(0, color=FC_CHARCOAL, linestyle='--', linewidth=1.2)
axes[1].axvline(residuals.mean(), color=FC_ORANGE, linewidth=1.5,
                label=f'Mean = {residuals.mean():.2f}')
axes[1].set_xlabel('Residual (Predicted \u2212 True)')
axes[1].set_ylabel('Count')
axes[1].set_title('NHITS \u2014 Residual Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 10 — Save 04d Outputs
# ══════════════════════════════════════════════════════════════════════════════

joblib.dump({
    'y_pred_nhits': y_pred_nhits,
    'y_val_last': y_true,
    'rmse': rmse,
    'mae': mae,
    'nasa_score': nasa,
    'hyperparameters': NHITS_PARAMS,
}, OUTPUTS_DIR / 'phase4d_nhits_predictions.pkl')

# No Optuna study \u2014 fixed literature-standard hyperparameters used.
joblib.dump(None, OUTPUTS_DIR / 'optuna_nhits_study.pkl')

print('Phase 4d outputs persisted:')
for f in sorted(OUTPUTS_DIR.glob('phase4d_*')) + sorted(OUTPUTS_DIR.glob('optuna_nhits*')):
    print(f'  {f.name}')
print(f'\n\u2714 Notebook 04d complete.')
